# Draft Utility Empirical Value Model

Purpose: prototype a defensible FOF8 draft utility target. This notebook estimates rating-to-value and position multipliers from observed market behavior, then uses rating trajectories to construct rookie-control utility targets and top-k board metrics.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, SplineTransformer, StandardScaler

ROOT = Path.cwd()
while not (ROOT / "fof8-ml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "fof8-core" / "src"))
sys.path.insert(0, str(ROOT / "fof8-ml" / "src"))

from fof8_core.loader import FOF8Loader  # noqa: E402

RAW_ROOT = ROOT / "fof8-gen" / "data" / "raw"
FEATURES_PATH = ROOT / "fof8-ml" / "data" / "processed" / "features.parquet"

pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(140)

## Configuration

This notebook now uses the same eight-universe, twenty-year exploratory slice as the audit notebook so the B1 recommendation is based on more than a single prototype universe. Expand to all leagues only if you need a slower confirmation pass.


In [ ]:
available_leagues = sorted(p.name for p in RAW_ROOT.glob("DRAFT*") if p.is_dir())

LEAGUES = available_leagues[:8]
MAX_YEARS = 20
VETERAN_MIN_EXPERIENCE = 5
CONTROL_YEARS = [1, 2, 3, 4]
CONTROL_DISCOUNTS = {1: 1.00, 2: 0.95, 3: 0.85, 4: 0.75}

LEAGUES


In [ ]:
def years_for(loader: FOF8Loader, max_years: int | None = MAX_YEARS) -> list[int]:
    years = list(range(loader.initial_sim_year, loader.final_sim_year + 1))
    return years if max_years is None else years[:max_years]

def load_market_panel(leagues: list[str]) -> pl.DataFrame:
    frames = []
    for league in leagues:
        loader = FOF8Loader(RAW_ROOT, league)
        years = years_for(loader)

        caps = (
            loader.scan_file("universe_info.csv")
            .filter(pl.col("Year").is_in(years))
            .filter(pl.col("Information") == "Salary Cap (in tens of thousands)")
            .select("Year", pl.col("Value/Round/Position").cast(pl.Float64).alias("Cap_10k"))
            .collect()
        )
        records = (
            loader.scan_file("player_record.csv")
            .filter(pl.col("Year").is_in(years))
            .select(
                "Year", "Player_ID", "Position", "Experience", "S_Games_Played", "S_Games_Started",
                "Salary_Year_1", "Bonus_Year_1",
            )
            .collect()
        )
        personal = (
            loader.scan_file("players_personal.csv")
            .filter(pl.col("Year").is_in(years))
            .select("Year", "Player_ID", "Current_Overall")
            .collect()
        )
        frame = (
            records.join(personal, on=["Year", "Player_ID"], how="inner")
            .join(caps, on="Year", how="left")
            .with_columns(
                pl.lit(league).alias("Universe"),
                (
                    (pl.col("Salary_Year_1") + pl.col("Bonus_Year_1"))
                    / pl.col("Cap_10k")
                ).alias("Annual_Cap_Share"),
            )
            .filter(pl.col("Annual_Cap_Share").is_finite())
        )
        frames.append(frame)
    return pl.concat(frames, how="vertical_relaxed")

market = load_market_panel(LEAGUES)
market.shape

## Estimate Rating-To-Market Value

This is an empirical estimate of how FOF8's market prices rating, position, and experience. We exclude early rookie-contract seasons to reduce wage-scale distortion.

In [ ]:
veteran_market = (
    market
    .filter(pl.col("Experience") >= VETERAN_MIN_EXPERIENCE)
    .filter(pl.col("Current_Overall") > 0)
    .filter(pl.col("Annual_Cap_Share") >= 0)
    .with_columns(pl.col("Annual_Cap_Share").clip(upper_bound=0.35).alias("Annual_Cap_Share_Clipped"))
)

model_df = veteran_market.select(
    "Current_Overall", "Experience", "Position", "Annual_Cap_Share_Clipped"
).drop_nulls().to_pandas()

X = model_df[["Current_Overall", "Experience", "Position"]]
y = np.log1p(model_df["Annual_Cap_Share_Clipped"].to_numpy())

preprocess = ColumnTransformer(
    transformers=[
        (
            "rating_spline",
            SplineTransformer(n_knots=8, degree=3, include_bias=False),
            ["Current_Overall"],
        ),
        ("numeric", StandardScaler(), ["Experience"]),
        ("position", OneHotEncoder(handle_unknown="ignore"), ["Position"]),
    ]
)

market_value_model = Pipeline([
    ("preprocess", preprocess),
    ("model", Ridge(alpha=1.0)),
])
market_value_model.fit(X, y)

pred = np.expm1(market_value_model.predict(X))
actual = model_df["Annual_Cap_Share_Clipped"].to_numpy()
{
    "rows": len(model_df),
    "mae_cap_share": mean_absolute_error(actual, pred),
    "r2_log_cap_share": r2_score(y, market_value_model.predict(X)),
}

In [ ]:
positions = sorted(model_df["Position"].dropna().unique())
rating_grid = pd.DataFrame(
    [(pos, overall, 6) for pos in positions for overall in range(20, 91)],
    columns=["Position", "Current_Overall", "Experience"],
)
rating_grid["Estimated_Cap_Share_Value"] = np.expm1(market_value_model.predict(rating_grid))
rating_curve = pl.from_pandas(rating_grid)
rating_curve.head(10)

In [ ]:
reference_overall = 65
position_multipliers = (
    rating_curve
    .filter(pl.col("Current_Overall") == reference_overall)
    .select("Position", "Estimated_Cap_Share_Value")
    .with_columns(
        (
            pl.col("Estimated_Cap_Share_Value")
            / pl.col("Estimated_Cap_Share_Value").median()
        ).alias("Market_Position_Multiplier")
    )
    .sort("Market_Position_Multiplier", descending=True)
)
position_multipliers

## Build Rookie-Control Utility Targets

This converts post-draft rating trajectories into target candidates. For Phase B1 we compare plain and discounted control-window rating summaries, then keep the market-derived utility column as a downstream evaluation target rather than the first modeling target.


In [ ]:
def load_rookies(leagues: list[str]) -> pl.DataFrame:
    frames = []
    for league in leagues:
        loader = FOF8Loader(RAW_ROOT, league)
        years = years_for(loader)
        pre = (
            loader.scan_file("player_information_pre_draft.csv")
            .filter(pl.col("Year").is_in(years))
            .select("Year", "Player_ID", "Position")
            .collect()
        )
        rookies = (
            loader.scan_file("rookies.csv")
            .filter(pl.col("Year").is_in(years))
            .select("Year", "Player_ID", "Position_Group", "Grade")
            .collect()
        )
        frames.append(
            rookies.join(pre, on=["Year", "Player_ID"], how="left")
            .with_columns(pl.lit(league).alias("Universe"))
            .rename({"Year": "Draft_Year"})
        )
    return pl.concat(frames, how="vertical_relaxed")

def load_exhibition_ratings(leagues: list[str]) -> pl.DataFrame:
    frames = []
    for league in leagues:
        loader = FOF8Loader(RAW_ROOT, league)
        years = years_for(loader)
        frame = (
            loader.scan_file("player_ratings_season_*.csv")
            .filter(pl.col("Year").is_in(years))
            .filter(pl.col("Scouting") == "Exhibition")
            .select("Year", "Player_ID", "Current_Overall")
            .with_columns(pl.lit(league).alias("Universe"))
            .collect()
        )
        frames.append(frame)
    return pl.concat(frames, how="vertical_relaxed")

rookies = load_rookies(LEAGUES)
ratings = load_exhibition_ratings(LEAGUES)
rookies.shape, ratings.shape

In [ ]:
rookie_trajectory = (
    rookies.join(ratings, on=["Universe", "Player_ID"], how="left")
    .with_columns((pl.col("Year") - pl.col("Draft_Year") + 1).alias("Career_Year"))
    .filter(pl.col("Career_Year").is_in(CONTROL_YEARS))
    .drop_nulls(["Current_Overall", "Position"])
)

trajectory_pd = rookie_trajectory.select(
    "Universe", "Draft_Year", "Player_ID", "Position", "Position_Group",
    "Grade", "Career_Year", "Current_Overall",
).to_pandas()
trajectory_pd["Experience"] = 6
trajectory_pd["Estimated_Cap_Share_Value"] = np.expm1(
    market_value_model.predict(
        trajectory_pd[["Current_Overall", "Experience", "Position"]]
    )
)
trajectory_pd["Control_Discount"] = trajectory_pd["Career_Year"].map(CONTROL_DISCOUNTS)
trajectory_pd["Discounted_Control_Utility"] = (
    trajectory_pd["Estimated_Cap_Share_Value"]
    * trajectory_pd["Control_Discount"]
)
trajectory_pd["Discounted_Current_Overall"] = (
    trajectory_pd["Current_Overall"]
    * trajectory_pd["Control_Discount"]
)

trajectory = pl.from_pandas(trajectory_pd)
trajectory.head(10)


In [ ]:
discount_weight_total = sum(CONTROL_DISCOUNTS.values())

utility_targets = (
    trajectory
    .group_by(["Universe", "Draft_Year", "Player_ID", "Position", "Position_Group", "Grade"])
    .agg(
        pl.col("Current_Overall").mean().alias("Control_Window_Mean_Current_Overall"),
        (pl.col("Discounted_Current_Overall").sum() / discount_weight_total)
            .alias("Control_Window_Discounted_Mean_Current_Overall"),
        pl.col("Discounted_Control_Utility").sum().alias("Control_Window_Utility"),
        pl.col("Estimated_Cap_Share_Value").max().alias("Peak_Empirical_Rating_Value"),
        pl.col("Current_Overall").top_k(3).mean().alias("Top3_Mean_Current_Overall_Rebuilt"),
        pl.col("Career_Year").n_unique().alias("Observed_Control_Years"),
    )
    .with_columns(
        (pl.col("Observed_Control_Years") == len(CONTROL_YEARS)).alias("Complete_Control_Window")
    )
)
utility_targets.describe()


## Board Metric Prototype

Use this to compare candidate target summaries against the market-derived control-window utility target. The key B1 question is whether the initial control-window modeling target should be the plain mean or the discounted mean.


In [ ]:
def mean_topk_utility_ratio(
    df: pl.DataFrame,
    *,
    score_col: str,
    utility_col: str,
    k: int,
    group_cols: list[str] = ["Universe", "Draft_Year"],
) -> float:
    values = []
    for _, group in df.group_by(group_cols):
        if len(group) == 0:
            continue
        k_eff = min(k, len(group))
        model_value = group.sort(score_col, descending=True).head(k_eff)[utility_col].sum()
        oracle_value = group.sort(utility_col, descending=True).head(k_eff)[utility_col].sum()
        if oracle_value and oracle_value > 0:
            values.append(float(model_value / oracle_value))
    return float(np.mean(values)) if values else 0.0

complete_window = utility_targets.filter(pl.col("Complete_Control_Window"))
candidate_rows = []
for score_col in [
    "Grade",
    "Control_Window_Mean_Current_Overall",
    "Control_Window_Discounted_Mean_Current_Overall",
    "Top3_Mean_Current_Overall_Rebuilt",
    "Peak_Empirical_Rating_Value",
]:
    corr = complete_window.select(score_col, "Control_Window_Utility").to_pandas().corr(method="spearman").iloc[0, 1]
    candidate_rows.append({
        "score_col": score_col,
        "spearman_to_control_utility": corr,
        "top16_utility_capture_ratio": mean_topk_utility_ratio(
            complete_window,
            score_col=score_col,
            utility_col="Control_Window_Utility",
            k=16,
        ),
        "top32_utility_capture_ratio": mean_topk_utility_ratio(
            complete_window,
            score_col=score_col,
            utility_col="Control_Window_Utility",
            k=32,
        ),
        "top64_utility_capture_ratio": mean_topk_utility_ratio(
            complete_window,
            score_col=score_col,
            utility_col="Control_Window_Utility",
            k=64,
        ),
    })

pl.DataFrame(candidate_rows).sort("top32_utility_capture_ratio", descending=True)


## Phase B1 Decisions

- The veteran-market curve is strong enough for a first-pass empirical utility baseline, but it should stay a downstream conversion layer rather than the first supervised target.
- `Control_Window_Mean_Current_Overall` is the better initial control-window modeling target. On the eight-universe slice it slightly beats the discounted mean on both Spearman correlation to utility and top-k utility capture.
- Keep `Control_Window_Discounted_Mean_Current_Overall` as an exported companion target for later board formulas and sensitivity checks.
- Keep empirical position multipliers as the default baseline, but expect config overrides once gameplay preference diverges from the market.
